# SciGuard: complete reproducible analysis and MIMIC-IV protocol
**Canonical notebook:** `SciGuard_JMLR_Reproducible_Analysis_MIMIC_IV.ipynb`  
**Major revision:** September 27, 2026 | Faustus Domebale Maale and Hui Yi

This notebook is the single current workflow. It contains the inferential calibration,
nested model refits, corrected missingness mechanisms, WDBC stress test, sample-size /
dependence / grid / bootstrap ablations, continuous extension, pilot-design example,
and credentialed MIMIC-IV execution path. All statistical functions are imported from
`src/sciguard`; none are redefined later in the notebook.

**Evidence status:** synthetic and WDBC analyses are reproducible using the bundled data.
MIMIC-IV credentialed-data execution and aggregate evaluation are included in the current revision; raw and row-level data are not distributed. A successful synthetic loader
test does not constitute clinical benchmark evidence. The revision is an applied-methods
framework, not a claim of a new Gaussian-bootstrap theorem or clinical safety.

## 1. Run controls and reproducibility
Unzip the entire package and open this notebook from its root. Install with
`python -m pip install -e ".[experiments,notebook]"`. Restart the kernel and run all cells.

`SCIGUARD_PROFILE=full` (default) recomputes every public/synthetic section. Use
`SCIGUARD_PROFILE=smoke` for a short execution check; smoke results have their own directory
and are never exported to manuscript tables. Full runs use 500 calibration replications,
200 nested refits, 100 missingness refits, 20,000 independent reference rows per refit,
2,000 bootstrap draws (5,000 for WDBC), and a fixed seed. Training is single-threaded
for reproducibility. Runtime depends on hardware.

Original notebooks and results are retained under `archive/pre_revision/` and are never
silently loaded by this workflow. The root notebook's displayed outputs are from its
recorded execution; rerunning replaces them with current outputs.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from threadpoolctl import threadpool_limits

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'src/sciguard/experiments.py').is_file()
             and (p / 'data/simulation_true_risks.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the unzipped SciGuard_JMLR_Target folder.')
sys.path.insert(0, str(ROOT / 'src'))
import sciguard
from sciguard.experiments import AnalysisRun, RunConfig, SECTIONS, LABELS
from sciguard import certify, certify_overlapping_groups, continuous_upper_envelope
from sciguard.plotting import (schematic, calibration_figure, sensitivity_figure,
    nested_figure, missingness_figure, wdbc_figure, continuous_figure, stress_figure)

PROFILE = os.environ.get('SCIGUARD_PROFILE', 'full')
if PROFILE not in {'full', 'smoke'}:
    raise ValueError('SCIGUARD_PROFILE must be full or smoke')
CONFIG = RunConfig() if PROFILE == 'full' else RunConfig.smoke()
THREAD_LIMIT = threadpool_limits(limits=1)
run = AnalysisRun(ROOT, CONFIG)
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
print('SciGuard', sciguard.__version__, '| profile:', PROFILE)
display(pd.DataFrame([run.manifest['config']]).T.rename(columns={0: 'Value'}))

## 2. Target, assumptions, and guarantees
Condition on the training/design data and a frozen pipeline $g$. For a prespecified
environment $e$, write $R_g(e)=E[L_e]$, $0\leq L_e\leq1$, and
$\mathcal A_\tau=\{e:R_g(e)\leq\tau\}$. A simultaneous band $U_e$ yields the inner estimate
$\widehat{\mathcal A}_\tau=\{e:U_e\leq\tau\}$.

The fundamental event is **simultaneous risk coverage**, not only containment at one
tolerance. If every $R_e\leq U_e$, then containment holds for **every** tolerance at once.
Thus inspecting several tolerances on the same fixed band does not invalidate this event.
Operational tolerances should nevertheless be prospectively justified. Changing the loss,
model, environment family, or certification method after seeing assurance outcomes needs
independent data or multiplicity control. Exact Holm here is a test at a fixed tolerance,
not a confidence band constructed uniformly over tolerances.

| Method | Scope |
|---|---|
| Multiplier | Asymptotic simultaneous band; requires appropriate row independence, nondegenerate population variances, and an adequate Gaussian approximation. Finite bootstrap draws add Monte Carlo error. |
| Normal Bonferroni | Multiplicity-corrected normal approximation; not finite-sample exact. |
| Exact Bonferroni | One-sided Clopper-Pearson bounds at $\alpha/K$ for binary losses; finite-sample familywise control. |
| Hoeffding | $\min(1,\widehat R_k+\sqrt{\log(K/\alpha)/(2n_k)})$; finite-sample control for independent bounded observations. |
| Exact Holm | Binomial lower-tail tests of $H_{0k}:R_k\geq\tau$, adjusted by Holm; finite-sample containment at a fixed tolerance. |
| Pointwise normal | Diagnostic comparator only; no familywise guarantee. |

For the multiplier route, zero empirical variance produces upper bound 1, never a false
zero-risk certificate. Exact binomial bounds can still certify a zero-error column if the
sample is sufficiently large. All moments use `ddof=0` consistently.

In [ ]:
fig = schematic(run)
display(fig); plt.close(fig)
toy_losses = np.zeros((100, 5))
display(pd.DataFrame({
    'Multiplier upper': certify(toy_losses, .10, n_boot=200, rng=1).upper,
    'Exact binomial upper': certify(toy_losses, .10, method='exact_bonferroni').upper
}))

### 2.1 Why shared transformations do not violate row independence
Let $Z_i$ be independent source observations and $V_i$ independent row-specific stress
randomness. With the design fixed, each vector
$\mathbf L_i=(h_1(Z_i,V_i),\ldots,h_K(Z_i,V_i))$ is a measurable function of one independent
row. Therefore the vectors are i.i.d.; their coordinates can be arbitrarily dependent,
including identical columns. One Gaussian multiplier is shared across each row's columns.

This argument fails if transformations normalize using the **assurance batch**, if repeated
patients are treated as independent, or if a shared random nuisance is not conditioned on.
The revised MAR/MNAR generators use fixed population-calibrated intercepts and row-local
randomness. The conditional-covariance sampler draws from the same Gaussian law as direct
row multipliers; this computational equivalence does not make the inferential approximation exact.

### 2.2 Evaluation metrics
With known truth: simultaneous coverage, containment, ARR (fraction of safe environments
certified), and UCP (unsafe fraction among certified environments). UCP is zero for an empty
certified set; ARR is undefined (`NaN`) if no environment is truly safe. Always report the
certified count so vacuous containment is visible. Confidence intervals for empirical
coverage/containment use Wilson intervals over independent Monte Carlo replications.

## 3. Calibration with exactly known risks
The supplied 25-cell risk surface is a calibration-only design. Correlated Bernoulli losses
are generated through a Gaussian copula; the model-fitting study is separate. Latent Gaussian
correlation is not the same as observed binary-loss correlation. No archived numerical targets
are imposed as assertions, and unfavorable results remain in the output.

In [ ]:
truth = pd.read_csv(ROOT / 'data/simulation_true_risks.csv')
assert len(truth) == 25 and truth.Risk.between(0, 1).all()
display(truth)
calibration_replications, calibration_summary = run.section('calibration', SECTIONS['calibration'])
display(calibration_summary[['method', 'simultaneous_coverage', 'simultaneous_coverage_lo',
    'simultaneous_coverage_hi', 'containment', 'ARR', 'ARR_mcse', 'UCP', 'n_certified']].round(4))
fig = calibration_figure(run); display(fig); plt.close(fig)

## 4. Nested fitted-pipeline experiment
Every replication draws fresh training data, fits logistic regression and histogram gradient
boosting, then uses separate source-calibration, assurance, and reference samples. Models share
the same generated rows and stress randomness for a paired comparison. The protected first
feature and all preprocessing are fixed before assurance. This changes the earlier experiment;
the revised numbers must not be compared as if only the random seed changed.

**Reference uncertainty:** 20,000 reference observations improve on the former 1,500, but their
means are still not exact true risks. Simultaneous two-sided exact binomial intervals use total
reference error 0.01 across both models in each refit. Cells whose intervals cross the tolerance
are ambiguous. `containment_lower` certifies containment only when all selected cells are
reference-confirmed safe; `containment_upper` rules it out only if a selected cell is
reference-confirmed unsafe. These are bounds on a containment indicator on the reference
coverage event, not confidence limits for a population coverage probability. Their averages
are reported alongside point-reference metrics and Wilson intervals. Reference coverage is
per refit, not simultaneous over every refit in the experiment.

In [ ]:
nested_replications, nested_summary = run.section('nested', SECTIONS['nested'])
display(nested_summary[['Model', 'method', 'reference_containment', 'reference_containment_lo',
    'reference_containment_hi', 'containment_lower', 'containment_upper',
    'reference_ARR', 'reference_UCP', 'reference_n_certified', 'ambiguous_environments']].round(4))
display(pd.read_csv(run.out / 'nested_fit_summary.csv').round(4))
fig = nested_figure(run); display(fig); plt.close(fig)

### 4.1 ATC is evaluated as a performance predictor
Average Thresholded Confidence with maximum-probability scores is calibrated on independent
labeled source data. It uses target confidence scores to predict target risk. We report MAE,
bias, and RMSE against the independent reference estimates. ATC is not placed in a table of
formal certification guarantees. Tied scores use a consistent strict threshold in calibration
and deployment. This is an ATC-MC implementation, not a newly corrected certifier.

In [ ]:
display(pd.read_csv(run.out / 'atc_prediction_summary.csv').round(4))

## 5. Corrected missingness mechanisms
All mechanisms keep feature 0 observed; severity is the missing fraction among the other
nine eligible features. MAR depends only on observed feature 0. MNAR depends on the feature
being hidden. Fixed logistic intercepts are calibrated under the known simulation distribution,
not renormalized on each assurance batch. Structured failure hides features 1-3 together while
maintaining the same marginal severity for all eligible features. The label equation and model
are unchanged across mechanisms, and masks are nested as severity increases.

The reference interval budget is 0.01 across all four mechanisms within each refit. MNAR and
block-failure scenarios are sensitivity assumptions; the results do not identify a real clinical
missingness mechanism. Realized missingness is exported for every environment.

In [ ]:
missingness_replications, missingness_summary = run.section('missingness', SECTIONS['missingness'])
display(missingness_summary[['Mechanism', 'method', 'reference_containment',
    'containment_lower', 'containment_upper', 'reference_ARR', 'reference_UCP',
    'ambiguous_environments']].round(4))
rates = pd.read_csv(run.out / 'missingness_environment_detail.csv')
display(rates.groupby(['Mechanism', 'Missingness']).realized_missingness.mean().unstack().round(4))
fig = missingness_figure(run); display(fig); plt.close(fig)

## 6. WDBC: three frozen models and grid resolution
The 65/35 stratified split (random state 42), train-only imputation/scaling, and shared feature
noise/missingness are applied to logistic regression, Extra Trees, and histogram gradient
boosting. This remains a small **semisynthetic stress test**, not a natural hospital/temporal
shift study. No ground-truth ARR, UCP, or repeated-sampling coverage is inferred from this
single real-data split.

Each model/design is a separate assurance family. Selecting the best model after comparing
these separate bands is not jointly protected. The nonuniform 36-cell primary grid and the
equally spaced 6-by-6 grid contain different missingness values; equal counts do not make
them identical designs. Grid fractions are not continuous areas.

In [ ]:
wdbc_detail, wdbc_summary = run.section('wdbc', SECTIONS['wdbc'])
display(wdbc_summary[wdbc_summary.design == 'primary'][['Model', 'method', 'n',
    'nominal_risk', 'certified_008', 'certified_010']].round(4))
display(wdbc_summary[wdbc_summary.design != 'primary'][['design', 'method', 'K',
    'certified_008', 'certified_fraction', 'critical']].round(4))
fig = wdbc_figure(run); display(fig); plt.close(fig)

## 7. Sample size, dependence, and the cost of multiplicity
The sensitivity study compares all certification methods on the same losses and reports
**paired** ARR differences, with Monte Carlo standard errors, relative to normal Bonferroni,
exact Bonferroni, and exact Holm. Any efficiency gain must be read alongside coverage and
containment. A wider certified region is not a valid improvement if its guarantee fails.

In [ ]:
sensitivity_replications, sensitivity_summary = run.section('sensitivity', SECTIONS['sensitivity'])
display(sensitivity_summary[['n', 'rho', 'method', 'simultaneous_coverage', 'containment',
    'ARR', 'UCP']].round(4))
display(pd.read_csv(run.out / 'paired_efficiency_gains.csv').round(4))
fig = sensitivity_figure(run); display(fig); plt.close(fig)

## 8. Bootstrap Monte Carlo error, runtime, and adverse regimes
For each fixed loss matrix, repeated seeds at B = 200, 1,000, and 5,000 measure bootstrap
critical-value variability and inference time. This is a timing experiment, not a guarantee
that one bootstrap size suffices in all settings. The direct sampler costs O(B n K); the
covariance route costs O(n K² + K³ + B K²), with different memory tradeoffs.

Additional studies use bounded continuous Beta losses, rare binary errors, and a larger
125-environment family. Exact binomial methods apply only to binary losses. Bounded loss
does not imply that the Gaussian approximation is accurate when events are rare. Adversarial
adaptive shifts, unbounded heavy-tailed losses, and high-dimensional continuous environment
geometry are not claimed to be addressed by these stress tests.

In [ ]:
bootstrap_replications, stress_replications = run.section('bootstrap_and_stress', SECTIONS['bootstrap_and_stress'])
display(pd.read_csv(run.out / 'bootstrap_runtime_summary.csv').round(5))
display(pd.read_csv(run.out / 'stress_summary.csv')[['scenario', 'method',
    'simultaneous_coverage', 'simultaneous_coverage_lo', 'simultaneous_coverage_hi',
    'containment', 'ARR']].round(4))
fig = stress_figure(run); display(fig); plt.close(fig)

## 9. Continuous SAR and the Lipschitz requirement
For a known global bound $|R(e)-R(e')|\le L_R\|e-e'\|$, the envelope
$U^{\mathrm{cont}}(e)=\min_k\{U_k+L_R\|e-e_k\|\}\wedge1$ covers all continuous risks
on the grid-coverage event. This is a standard consequence of simultaneous coverage and
smoothness, not a new bootstrap theorem.

The bilinear example $R(m,s)=.24+.12m+.10s+.05ms$ on $[0,.4]\times[0,.6]$ has the analytic
bound $L_R=\sqrt{.15^2+.12^2}$. Dense-mesh containment/recall are numerical diagnostics,
not a numerical proof over every point. The proof supplies continuous coverage when the
bound is valid. Files also record sensitivity to half and twice the analytic bound; the
half-bound scenario is not asserted valid.

**Do not estimate a global Lipschitz upper bound by the maximum observed grid slope.**
Even independent pilot slopes do not control unseen behavior between grid points. Without
external smoothness information, report finite-grid results. If a separately justified random
upper bound fails with probability delta, the combined coverage budget is alpha + delta.

In [ ]:
continuous_replications, continuous_summary = run.section('continuous', SECTIONS['continuous'])
display(continuous_summary[['method', 'grid_coverage', 'containment', 'ARR', 'UCP']].round(4))
fig = continuous_figure(run); display(fig); plt.close(fig)

## 10. Environment design and independent pilot refinement
Choose the scientific loss and deployment axes; set ranges from external knowledge;
freeze preprocessing and the candidate family; choose sample size for a meaningful margin;
use a pilot to prioritize a finite family; then evaluate that frozen family on **independent**
assurance rows. The example selects grid points near a pilot-estimated boundary plus coordinate
extremes. Its exact-binomial guarantee is conditional on the selected family, not on the
entire continuous rectangle.

The Hoeffding planning inequality $n\ge\log(K/\alpha)/(2\epsilon^2)$ ensures the additive
bound width is at most epsilon. It does not ensure a desired recall or a desired probability
of certification, which also depends on the true margin and sampling variation.

Repeated deployment checks are not time-uniform by default. A simple, conservative extension
allocates $\alpha_t=\alpha/[t(t+1)]$ to each round with valid conditional finite-sample
inference and fresh assurance data. The union bound then controls any false certification
over all rounds. This is standard alpha spending, not a newly optimal sequential SAR method.

In [ ]:
pilot_replications, planning_table = run.section('pilot_design', SECTIONS['pilot_design'])
display(pd.read_csv(run.out / 'pilot_design_summary.csv').round(4))
display(planning_table)

## 11. MIMIC-IV protocol: prerequisites and temporal corrections
The attached archive contains no credentialed MIMIC tables and no successful MIMIC results.
The earlier notebook has a recorded `FileNotFoundError`. This section is a fully implemented
**local execution protocol**, not evidence of completed external validation.

Use the actual dataset version in `config/mimic_protocol.json`, supply a prospectively
justified `tau` and `tau_rationale`, and set `MIMIC_IV_PATH` and `SCIGUARD_RUN_MIMIC=1`.
The example configuration is for v3.1; its tolerance is deliberately unset. No universal
clinical tolerance is inferred from an example value such as 0.35.

**Cohort:** first eligible adult hospitalization per patient; in-hospital mortality outcome;
admission-table demographic/admission variables; train-only preprocessing; 0.5 decision
threshold. Patient identifiers, mortality flags, discharge/death dates, and length of stay
are excluded from predictors. Availability of the admission-table fields at the intended
decision time still requires an author/data audit; this is a limited clinical baseline.

**Temporal design:** `anchor_year_group` describes the patient's anchor year, not every
admission. The loader adds `year(admittime) - anchor_year` to its endpoints, uses configurable
one-year padding for approximate year alignment, and restricts intervals to the declared
dataset coverage. Source intervals end by the cutoff; deployment intervals begin after it;
overlapping intervals are excluded. Deployment labels are approximate admission intervals
and may overlap one another in calendar time. They are not exact admission years or sites.
Top-coded age 91 is retained as a category/provenance indicator rather than interpreted as
exact age or used to exclude the oldest patients.

Official date guidance: https://physionet.org/content/mimiciv/3.1/ (accessed September 27, 2026).
The one-year padding is a conservative analysis choice, not an official exact-date guarantee.

### 11.1 Dependence, class imbalance, and natural missingness
Within each domain, missingness levels reuse the same patients. The common-row influence
vector is $\widehat\psi_{i,d,m}=G_{id}(L_i^{(m)}-\widehat R_{d,m})/\widehat p_d$.
Overlapping groups and unequal counts are supported without truncating or inventing pairings.
The multiplier ratio-estimator route assumes nonvanishing group probabilities as well as
appropriate row sampling. Exact group-conditional binomial and Hoeffding bands are supplied.

All models, missingness levels, and domains belong to **one** simultaneous family. Small
domains remain visible and unvalidated. An always-survive baseline, mortality prevalence,
sensitivity/specificity, balanced accuracy, AUROC, and AUPRC reveal when low 0-1 risk is
uninformative under class imbalance. These diagnostic metrics do not inherit the 0-1-risk
certification guarantee. Native feature missingness is reported separately from added MCAR
and total missingness. Adding MCAR does not establish robustness to clinical MAR/MNAR.

Only aggregate result tables are saved. No row-level cohort, patient identifiers, or fitted
model is exported. Source file hashes and the frozen configuration support local provenance.
One admission per patient removes repeated-patient rows; it does not prove exchangeability
or eliminate hospital/time clustering. Actual-data validation remains required.

In [ ]:
RUN_MIMIC = os.environ.get('SCIGUARD_RUN_MIMIC', '0') == '1'

In [ ]:
import os
from pathlib import Path

# Set these variables in your local environment before executing MIMIC-IV cells.
RUN_MIMIC = os.environ.get("SCIGUARD_RUN_MIMIC", "0") == "1"
MIMIC_ROOT = Path(os.environ.get("MIMIC_IV_PATH", ROOT / "data" / "mimic-iv")).expanduser()

print("MIMIC execution requested:", RUN_MIMIC)
print("Configured MIMIC root exists:", MIMIC_ROOT.exists())


In [ ]:
print("Admissions files:")
print(list((MIMIC_ROOT / "hosp").glob("admissions*")))

print("\nPatients files:")
print(list((MIMIC_ROOT / "hosp").glob("patients*")))

print("\nICU stay files:")
print(list((MIMIC_ROOT / "icu").glob("icustays*")))

In [ ]:
from sciguard.mimic import run_mimic

MIMIC_CONFIG_PATH = ROOT / 'config/mimic_protocol.json'
MIMIC_CONFIG = json.loads(MIMIC_CONFIG_PATH.read_text())

# Prespecify the acceptable risk threshold BEFORE running MIMIC
MIMIC_CONFIG["tau"] = 0.35

MIMIC_CONFIG["tau_rationale"] = (
    "We prespecify tau = 0.35 as an operational maximum tolerable "
    "0-1 classification error for evaluating SciGuard under MIMIC-IV "
    "deployment shifts. This threshold is fixed before examining the "
    "MIMIC-IV deployment results and is not tuned using the outcomes."
)

RUN_MIMIC = os.environ.get('SCIGUARD_RUN_MIMIC', '0') == '1'

MIMIC_ROOT = Path(
    os.environ.get(
        'MIMIC_IV_PATH',
        str(ROOT / 'data/mimic-iv')
    )
).expanduser()

print("RUN_MIMIC:", RUN_MIMIC)
print("MIMIC_ROOT:", MIMIC_ROOT)

if RUN_MIMIC:
    if not MIMIC_ROOT.is_dir():
        raise FileNotFoundError(
            'Requested MIMIC execution, but MIMIC_IV_PATH '
            'is not an existing dataset directory.'
        )

    mimic_tables, mimic_manifest = run_mimic(
        MIMIC_ROOT,
        MIMIC_CONFIG,
        ROOT / 'results/mimic_local'
    )

    run.manifest['external_benchmark'] = (
        'completed_credentialed_data'
    )

    display(mimic_tables['mimic_cohort_flow'])
    display(mimic_tables['mimic_results'].round(4))
    display(mimic_tables['mimic_diagnostics'].round(4))
    display(mimic_tables['mimic_natural_missingness'].round(4))

    from sciguard.plotting import mimic_decision_map

    mimic_figure = mimic_decision_map(
        mimic_tables['mimic_results'],
        ROOT / 'figures/mimic_local'
    )

    display(mimic_figure)
    plt.close(mimic_figure)

run.flush()

In [ ]:
# ============================================================
# SAVE FINAL MIMIC-IV RESULTS FOR THE MANUSCRIPT
# ============================================================

import json
from pathlib import Path

MIMIC_RESULTS_DIR = ROOT / "results" / "mimic_local"
MIMIC_FIG_DIR = ROOT / "figures" / "mimic_local"

MIMIC_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MIMIC_FIG_DIR.mkdir(parents=True, exist_ok=True)

# Save all result tables
mimic_tables["mimic_cohort_flow"].to_csv(
    MIMIC_RESULTS_DIR / "mimic_cohort_flow.csv",
    index=False
)

mimic_tables["mimic_results"].to_csv(
    MIMIC_RESULTS_DIR / "mimic_results.csv",
    index=False
)

mimic_tables["mimic_diagnostics"].to_csv(
    MIMIC_RESULTS_DIR / "mimic_diagnostics.csv",
    index=False
)

mimic_tables["mimic_natural_missingness"].to_csv(
    MIMIC_RESULTS_DIR / "mimic_natural_missingness.csv",
    index=False
)

# Save benchmark manifest
with open(
    MIMIC_RESULTS_DIR / "mimic_manifest.json",
    "w"
) as f:
    json.dump(
        mimic_manifest,
        f,
        indent=2,
        default=str
    )

# Save publication-quality figure
mimic_figure.savefig(
    MIMIC_FIG_DIR / "mimic_sar_decision_map.pdf",
    bbox_inches="tight"
)

mimic_figure.savefig(
    MIMIC_FIG_DIR / "mimic_sar_decision_map.png",
    dpi=300,
    bbox_inches="tight"
)

print("Saved MIMIC-IV manuscript outputs:")
print(MIMIC_RESULTS_DIR)
print(MIMIC_FIG_DIR)

In [ ]:
print(mimic_tables["mimic_results"].columns.tolist())

display(
    mimic_tables["mimic_results"].round(4)
)

In [ ]:
mimic_tables["mimic_results"].round(4).to_latex(
    MIMIC_RESULTS_DIR / "mimic_results_table.tex",
    index=False,
    escape=False
)

In [ ]:
display(mimic_tables["mimic_results"].round(4))
display(mimic_tables["mimic_diagnostics"].round(4))
display(mimic_tables["mimic_cohort_flow"])

In [ ]:
# ============================================================
# MIMIC-IV SECONDARY CLASS-SENSITIVE SAR ANALYSIS
#
# Reviewer-requested secondary specification:
#   Loss        = false-negative risk among mortality cases
#   Requirement = sensitivity >= 80%
#   tau_FN      = 0.20
#
# The learned-model classification thresholds are calibrated
# ONLY on the independent source holdout and are then frozen
# before deployment evaluation.
#
# Primary inference: exact Clopper-Pearson Bonferroni
# Secondary check : exact Holm testing
# ============================================================

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from scipy.stats import binom

from sciguard.mimic import (
    load_cohort,
    build_model,
    FEATURES,
)

from sciguard.groups import certify_overlapping_groups
from sciguard.baselines import holm_reject


# ------------------------------------------------------------
# 1. FREEZE THE SECONDARY SPECIFICATION BEFORE COMPUTING
#    THE NEW DEPLOYMENT CERTIFICATES
# ------------------------------------------------------------

TAU_FN = 0.20
TARGET_SOURCE_SENSITIVITY = 0.80
ALPHA_FN = 0.05

# Exact inference remains valid in sparse-event settings.
# The implementation requires at least 2 positive cases
# for a group to be marked eligible.
MIN_POSITIVE_COUNT = 2

FN_RESULTS_DIR = ROOT / "results" / "mimic_local"
FN_FIG_DIR = ROOT / "figures" / "mimic_class_sensitive"

FN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FN_FIG_DIR.mkdir(parents=True, exist_ok=True)

secondary_protocol = {
    "analysis": "reviewer_requested_secondary_class_sensitive_MIMIC_IV",
    "dataset": "MIMIC-IV v3.1",
    "loss": "false_negative_risk_conditional_on_Y_equals_1",
    "tau_fn": TAU_FN,
    "tau_interpretation": (
        "Maximum tolerable false-negative risk of 0.20, "
        "equivalent to a deployment sensitivity requirement of at least 0.80."
    ),
    "source_threshold_target_sensitivity": TARGET_SOURCE_SENSITIVITY,
    "threshold_rule": (
        "For each learned model, choose the largest classification threshold "
        "on the independent source holdout that attains sensitivity >= 0.80. "
        "Freeze that threshold before deployment evaluation."
    ),
    "primary_inference": "exact_Clopper_Pearson_Bonferroni",
    "secondary_inference": "exact_Holm",
    "alpha": ALPHA_FN,
    "minimum_positive_count_for_eligibility": MIN_POSITIVE_COUNT,
    "seed": int(MIMIC_CONFIG["seed"]),
    "note": (
        "This is a reviewer-requested secondary analysis added after the "
        "primary MIMIC analysis. The secondary loss, tolerance, and threshold "
        "calibration rule are fixed before recomputing the class-sensitive "
        "deployment certificates. The 80% sensitivity target is an operational "
        "analysis requirement and is not claimed to be a clinical standard."
    ),
}

SECONDARY_PROTOCOL_PATH = (
    FN_RESULTS_DIR / "mimic_class_sensitive_protocol.json"
)

SECONDARY_PROTOCOL_PATH.write_text(
    json.dumps(secondary_protocol, indent=2)
)

print("Secondary protocol frozen at:")
print(SECONDARY_PROTOCOL_PATH)

print("\nFalse-negative tolerance:", TAU_FN)
print(
    "Equivalent sensitivity requirement:",
    1 - TAU_FN
)


# ------------------------------------------------------------
# 2. LOAD THE SAME PRESPECIFIED MIMIC COHORT
# ------------------------------------------------------------

mimic_df, _, _ = load_cohort(
    MIMIC_ROOT,
    MIMIC_CONFIG
)

source = mimic_df[
    mimic_df["split"] == "source"
].copy()

target = mimic_df[
    mimic_df["split"] == "deployment"
].copy()

train, source_holdout = train_test_split(
    source,
    test_size=0.20,
    stratify=source["mortality"],
    random_state=MIMIC_CONFIG["seed"],
)

# Safety checks
assert set(train["subject_id"]).isdisjoint(
    set(source_holdout["subject_id"])
)

assert set(source["subject_id"]).isdisjoint(
    set(target["subject_id"])
)

print("\nCohort sizes")
print("------------")
print("Training:", len(train))
print("Source holdout:", len(source_holdout))
print("Deployment:", len(target))

print(
    "Source-holdout deaths:",
    int(source_holdout["mortality"].sum())
)

print(
    "Deployment deaths:",
    int(target["mortality"].sum())
)


# ------------------------------------------------------------
# 3. HELPER:
#    largest threshold giving >= target sensitivity
# ------------------------------------------------------------

def threshold_for_target_sensitivity(
    probabilities,
    labels,
    target_sensitivity=0.80,
):
    probabilities = np.asarray(probabilities, dtype=float)
    labels = np.asarray(labels, dtype=int)

    positive_scores = probabilities[labels == 1]

    if len(positive_scores) == 0:
        raise ValueError(
            "No positive outcomes in source holdout."
        )

    candidates = np.unique(positive_scores)

    sensitivities = np.array([
        np.mean(positive_scores >= threshold)
        for threshold in candidates
    ])

    valid = sensitivities >= target_sensitivity

    if not np.any(valid):
        raise ValueError(
            "Unable to attain requested source sensitivity."
        )

    # Largest threshold satisfying the target.
    # This gives the most specific classifier among thresholds
    # that still meet the source sensitivity requirement.
    threshold = candidates[valid][-1]

    achieved_sensitivity = np.mean(
        positive_scores >= threshold
    )

    return float(threshold), float(achieved_sensitivity)


# ------------------------------------------------------------
# 4. DEFINE REAL DEPLOYMENT DOMAINS
# ------------------------------------------------------------

domains = sorted(
    target["TemporalDomain"].unique()
)

domain_matrix = np.column_stack([
    target["TemporalDomain"]
    .eq(domain)
    .to_numpy()
    for domain in domains
])

y_target = (
    target["mortality"]
    .to_numpy(dtype=int)
)

X_target = (
    target[FEATURES]
    .reset_index(drop=True)
)

# Conditional groups:
# domain AND actual mortality Y=1
positive_indicator = (
    y_target == 1
).astype(float)

positive_group_matrix = (
    domain_matrix
    * positive_indicator[:, None]
)

print("\nPositive cases by deployment domain")
print("-----------------------------------")

positive_counts_by_domain = (
    positive_group_matrix.sum(axis=0).astype(int)
)

for domain, n_positive in zip(
    domains,
    positive_counts_by_domain
):
    print(
        f"{domain}: {n_positive} mortality cases"
    )


# ------------------------------------------------------------
# 5. USE EXACTLY THE SAME ADDED-MISSINGNESS CONSTRUCTION
#    AS THE PRIMARY MIMIC ANALYSIS
# ------------------------------------------------------------

rng = np.random.default_rng(
    MIMIC_CONFIG["seed"]
)

uniforms = rng.random(
    X_target.shape
)

# age_topcoded is provenance, not a measured feature;
# preserve it exactly as in run_mimic().
uniforms[
    :,
    FEATURES.index("age_topcoded")
] = 1.0


# ------------------------------------------------------------
# 6. FIT MODELS AND CALIBRATE CLASSIFICATION THRESHOLDS
#    USING ONLY THE SOURCE HOLDOUT
# ------------------------------------------------------------

model_names = (
    list(MIMIC_CONFIG["models"])
    + ["Always survive"]
)

threshold_rows = []

# Each column will contain:
# I(prediction == 0)
#
# The positive-group matrix then conditions this loss on Y=1.
fn_loss_columns = []
column_meta = []

for model_name in model_names:

    if model_name == "Always survive":

        model = None
        threshold = np.nan

        source_sensitivity = 0.0
        source_specificity = 1.0

        print(
            "\nAlways survive:"
            " sensitivity = 0 by construction"
        )

    else:

        model = build_model(
            model_name,
            MIMIC_CONFIG["seed"],
        )

        model.fit(
            train[FEATURES],
            train["mortality"],
        )

        source_prob = model.predict_proba(
            source_holdout[FEATURES]
        )[:, 1]

        threshold, source_sensitivity = (
            threshold_for_target_sensitivity(
                source_prob,
                source_holdout["mortality"],
                TARGET_SOURCE_SENSITIVITY,
            )
        )

        source_pred = (
            source_prob >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            source_holdout["mortality"],
            source_pred,
            labels=[0, 1],
        ).ravel()

        source_specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        )

        print(
            f"\n{model_name}"
        )

        print(
            "  calibrated threshold:",
            round(threshold, 6)
        )

        print(
            "  source sensitivity:",
            round(source_sensitivity, 4)
        )

        print(
            "  source specificity:",
            round(source_specificity, 4)
        )

    threshold_rows.append({
        "Model": model_name,
        "TargetSourceSensitivity":
            TARGET_SOURCE_SENSITIVITY,
        "FrozenThreshold": threshold,
        "AchievedSourceSensitivity":
            source_sensitivity,
        "SourceSpecificity":
            source_specificity,
    })


    # --------------------------------------------------------
    # Evaluate the frozen pipeline under deployment
    # missingness conditions
    # --------------------------------------------------------

    for missingness in MIMIC_CONFIG[
        "missingness_levels"
    ]:

        X_missing = X_target.mask(
            uniforms < missingness
        )

        if model is None:

            # Always-survive predicts 0 for everybody.
            pred = np.zeros(
                len(y_target),
                dtype=int,
            )

        else:

            deployment_prob = (
                model.predict_proba(
                    X_missing
                )[:, 1]
            )

            pred = (
                deployment_prob >= threshold
            ).astype(int)

        # False-negative loss indicator.
        #
        # This is defined for all rows; the positive group
        # matrix restricts estimation to actual deaths.
        fn_loss = (
            pred == 0
        ).astype(float)

        fn_loss_columns.append(
            fn_loss
        )

        column_meta.append(
            (
                model_name,
                float(missingness),
            )
        )


threshold_table = pd.DataFrame(
    threshold_rows
)

fn_losses = np.column_stack(
    fn_loss_columns
)

print(
    "\nFalse-negative loss matrix:",
    fn_losses.shape
)


# ------------------------------------------------------------
# 7. PRIMARY INFERENCE:
#    EXACT CLOPPER-PEARSON + BONFERRONI
#
# This is deliberately primary because the outcome is binary
# and positive-event counts are much smaller than total n.
# ------------------------------------------------------------

exact_result = certify_overlapping_groups(
    fn_losses,
    positive_group_matrix,
    tau=TAU_FN,
    alpha=ALPHA_FN,
    B=MIMIC_CONFIG["bootstrap_draws"],
    seed=MIMIC_CONFIG["seed"] + 8001,
    min_group_size=MIN_POSITIVE_COUNT,
    method="exact_bonferroni",
)


# ------------------------------------------------------------
# 8. SECONDARY EXACT HOLM TEST
# ------------------------------------------------------------

positive_sizes = np.repeat(
    positive_group_matrix.sum(axis=0),
    len(column_meta),
).astype(int)

fn_counts = (
    positive_group_matrix.T
    @ fn_losses
).ravel().astype(int)

holm_eligible = (
    positive_sizes
    >= MIN_POSITIVE_COUNT
)

holm_pvalues = np.ones(
    len(positive_sizes),
    dtype=float,
)

holm_pvalues[
    holm_eligible
] = binom.cdf(
    fn_counts[holm_eligible],
    positive_sizes[holm_eligible],
    TAU_FN,
)

holm_certified = (
    holm_reject(
        holm_pvalues,
        ALPHA_FN,
    )
    & holm_eligible
)


# ------------------------------------------------------------
# 9. BUILD LONG-FORM RESULT TABLE
# ------------------------------------------------------------

rows = []

n_columns = len(column_meta)

for i, domain in enumerate(domains):

    for j, (
        model_name,
        missingness,
    ) in enumerate(column_meta):

        k = (
            i * n_columns
            + j
        )

        n_pos = int(
            positive_sizes[k]
        )

        n_fn = int(
            fn_counts[k]
        )

        risk = (
            n_fn / n_pos
            if n_pos > 0
            else np.nan
        )

        sensitivity = (
            1 - risk
            if np.isfinite(risk)
            else np.nan
        )

        # Exact Bonferroni result
        rows.append({
            "TemporalDomain": domain,
            "Model": model_name,
            "Missingness": missingness,
            "method": "exact_bonferroni",
            "positive_n": n_pos,
            "false_negatives": n_fn,
            "fn_risk": risk,
            "sensitivity": sensitivity,
            "upper_fn_risk":
                exact_result.upper[k],
            "certified":
                bool(exact_result.certified[k]),
            "eligible":
                bool(exact_result.eligible[k]),
            "tau_fn": TAU_FN,
            "required_sensitivity":
                1 - TAU_FN,
            "alpha": ALPHA_FN,
        })

        # Holm result
        rows.append({
            "TemporalDomain": domain,
            "Model": model_name,
            "Missingness": missingness,
            "method": "holm",
            "positive_n": n_pos,
            "false_negatives": n_fn,
            "fn_risk": risk,
            "sensitivity": sensitivity,
            "upper_fn_risk": np.nan,
            "certified":
                bool(holm_certified[k]),
            "eligible":
                bool(holm_eligible[k]),
            "tau_fn": TAU_FN,
            "required_sensitivity":
                1 - TAU_FN,
            "alpha": ALPHA_FN,
        })


class_sensitive_results = pd.DataFrame(
    rows
)


# ------------------------------------------------------------
# 10. SAVE ALL RESULTS
# ------------------------------------------------------------

threshold_table.to_csv(
    FN_RESULTS_DIR
    / "mimic_class_sensitive_thresholds.csv",
    index=False,
)

class_sensitive_results.to_csv(
    FN_RESULTS_DIR
    / "mimic_class_sensitive_results.csv",
    index=False,
)


# ------------------------------------------------------------
# 11. DISPLAY THE IMPORTANT TABLES
# ------------------------------------------------------------

print(
    "\n===================================================="
)
print(
    "SOURCE-HOLDOUT THRESHOLD CALIBRATION"
)
print(
    "===================================================="
)

display(
    threshold_table.round(4)
)


exact_table = (
    class_sensitive_results[
        class_sensitive_results[
            "method"
        ] == "exact_bonferroni"
    ]
    .copy()
)

print(
    "\n===================================================="
)
print(
    "MIMIC-IV CLASS-SENSITIVE SAR RESULTS"
)
print(
    "Primary method: exact Clopper-Pearson Bonferroni"
)
print(
    "===================================================="
)

display(
    exact_table.round(4)
)


# ------------------------------------------------------------
# 12. COMPACT SUMMARY FOR THE MANUSCRIPT
# ------------------------------------------------------------

summary = (
    exact_table
    .groupby(
        ["Model", "Missingness"],
        as_index=False,
    )
    .agg(
        eligible_environments=(
            "eligible",
            "sum",
        ),
        certified_environments=(
            "certified",
            "sum",
        ),
        min_sensitivity=(
            "sensitivity",
            "min",
        ),
        max_sensitivity=(
            "sensitivity",
            "max",
        ),
        max_upper_fn_risk=(
            "upper_fn_risk",
            "max",
        ),
        min_positive_n=(
            "positive_n",
            "min",
        ),
    )
)

summary[
    "certified_fraction"
] = np.where(
    summary[
        "eligible_environments"
    ] > 0,
    summary[
        "certified_environments"
    ]
    / summary[
        "eligible_environments"
    ],
    np.nan,
)

summary.to_csv(
    FN_RESULTS_DIR
    / "mimic_class_sensitive_summary.csv",
    index=False,
)

print(
    "\n===================================================="
)
print(
    "COMPACT MANUSCRIPT SUMMARY"
)
print(
    "===================================================="
)

display(
    summary.round(4)
)


# ------------------------------------------------------------
# 13. OVERALL COUNTS
# ------------------------------------------------------------

eligible_exact = exact_table[
    exact_table["eligible"]
]

print(
    "\n===================================================="
)
print(
    "SECONDARY MIMIC-IV KEY NUMBERS"
)
print(
    "===================================================="
)

print(
    "Total exact-family conditions:",
    len(exact_table),
)

print(
    "Eligible conditions:",
    int(
        exact_table[
            "eligible"
        ].sum()
    ),
)

print(
    "Certified conditions:",
    int(
        exact_table[
            "certified"
        ].sum()
    ),
)

if len(eligible_exact) > 0:

    print(
        "Minimum observed sensitivity:",
        round(
            eligible_exact[
                "sensitivity"
            ].min(),
            4,
        ),
    )

    print(
        "Maximum observed sensitivity:",
        round(
            eligible_exact[
                "sensitivity"
            ].max(),
            4,
        ),
    )

    print(
        "Maximum exact upper FN risk:",
        round(
            eligible_exact[
                "upper_fn_risk"
            ].max(),
            4,
        ),
    )


# ------------------------------------------------------------
# 14. CLASS-SENSITIVE ASSURANCE MAP
# ------------------------------------------------------------

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

plot_data = exact_table.copy()

plot_models = list(
    dict.fromkeys(
        plot_data["Model"]
    )
)

plot_domains = sorted(
    plot_data[
        "TemporalDomain"
    ].unique()
)

plot_levels = sorted(
    plot_data[
        "Missingness"
    ].unique()
)

fig, axes = plt.subplots(
    1,
    len(plot_models),
    figsize=(
        4.3 * len(plot_models),
        max(
            4.8,
            0.48 * len(plot_domains) + 2,
        ),
    ),
    squeeze=False,
    layout="constrained",
)

# gray / red / yellow / green
cmap = ListedColormap([
    "#bcbcbc",
    "#d97b70",
    "#f0c56e",
    "#65ac92",
])

for ax, model_name in zip(
    axes[0],
    plot_models,
):

    states = np.zeros(
        (
            len(plot_domains),
            len(plot_levels),
        )
    )

    annotations = np.empty(
        states.shape,
        dtype=object,
    )

    for i, domain in enumerate(
        plot_domains
    ):

        for j, missingness in enumerate(
            plot_levels
        ):

            row = plot_data[
                (
                    plot_data[
                        "Model"
                    ] == model_name
                )
                & (
                    plot_data[
                        "TemporalDomain"
                    ] == domain
                )
                & np.isclose(
                    plot_data[
                        "Missingness"
                    ],
                    missingness,
                )
            ].iloc[0]

            if not row["eligible"]:

                states[i, j] = 0

                annotations[
                    i, j
                ] = (
                    f"n+={int(row['positive_n'])}"
                    "\ninsufficient"
                )

            elif row["certified"]:

                states[i, j] = 3

                annotations[
                    i, j
                ] = (
                    f"U={row['upper_fn_risk']:.2f}"
                    f"\nSens={row['sensitivity']:.2f}"
                )

            elif (
                row["fn_risk"]
                <= TAU_FN
            ):

                states[i, j] = 2

                annotations[
                    i, j
                ] = (
                    f"U={row['upper_fn_risk']:.2f}"
                    f"\nSens={row['sensitivity']:.2f}"
                )

            else:

                states[i, j] = 1

                annotations[
                    i, j
                ] = (
                    f"R={row['fn_risk']:.2f}"
                    f"\nSens={row['sensitivity']:.2f}"
                )

    ax.imshow(
        states,
        cmap=cmap,
        vmin=0,
        vmax=3,
        aspect="auto",
    )

    ax.set_xticks(
        range(
            len(plot_levels)
        )
    )

    ax.set_xticklabels([
        f"{m:.0%}"
        for m in plot_levels
    ])

    ax.set_yticks(
        range(
            len(plot_domains)
        )
    )

    ax.set_yticklabels(
        plot_domains
    )

    ax.set_xlabel(
        "Added MCAR feature missingness"
    )

    ax.set_title(
        model_name
    )

    for i, j in np.ndindex(
        states.shape
    ):

        ax.text(
            j,
            i,
            annotations[i, j],
            ha="center",
            va="center",
            fontsize=7,
        )


axes[
    0, 0
].set_ylabel(
    "Approximate admission interval"
)

fig.suptitle(
    "MIMIC-IV class-sensitive assurance map\n"
    "False-negative risk; "
    r"$\tau_{\mathrm{FN}}=0.20$ "
    "(required sensitivity ≥ 0.80)"
)

fig.legend(
    handles=[
        Patch(
            color=cmap.colors[0],
            label="Insufficient positive cases",
        ),
        Patch(
            color=cmap.colors[1],
            label="Observed FN risk > tolerance",
        ),
        Patch(
            color=cmap.colors[2],
            label="Observed FN risk ≤ tolerance; uncertified",
        ),
        Patch(
            color=cmap.colors[3],
            label="Certified",
        ),
    ],
    loc="outside lower center",
    ncol=2,
    fontsize=8,
)

FIG_PNG = (
    FN_FIG_DIR
    / "mimic_class_sensitive_sar.png"
)

FIG_PDF = (
    FN_FIG_DIR
    / "mimic_class_sensitive_sar.pdf"
)

fig.savefig(
    FIG_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIG_PDF,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# 15. FINAL FILE LOCATIONS
# ------------------------------------------------------------

print(
    "\nSaved:"
)

print(
    FN_RESULTS_DIR
    / "mimic_class_sensitive_protocol.json"
)

print(
    FN_RESULTS_DIR
    / "mimic_class_sensitive_thresholds.csv"
)

print(
    FN_RESULTS_DIR
    / "mimic_class_sensitive_results.csv"
)

print(
    FN_RESULTS_DIR
    / "mimic_class_sensitive_summary.csv"
)

print(
    FIG_PNG
)

print(
    FIG_PDF
)

## 12. Integrity checks, manuscript export, and interpretation
Assertions below check structural consistency and data provenance, not agreement with a
preferred numerical answer. Manuscript tables/macros are generated directly from the
completed full-run CSVs. Bootstrap seed changes can change numerical results; versions and
random seeds are recorded. The `archive/` results are historical and do not feed current tables.

The software checks and synthetic MIMIC fixture test are available through
`python -m unittest discover -s tests -v`. The real MIMIC data path cannot be validated by a
synthetic fixture. An absent external benchmark remains an open publication limitation.

In [ ]:
expected_sections = set(SECTIONS)
assert set(run.manifest['sections']) == expected_sections
assert all(v['status'] == 'completed' for v in run.manifest['sections'].values())
assert len(calibration_replications) == CONFIG.calibration_reps * 6
assert len(nested_replications) == CONFIG.nested_reps * 2 * 5
assert len(missingness_replications) == CONFIG.missingness_reps * 4 * 3
assert (nested_replications.containment_lower <= nested_replications.reference_containment).all()
assert (nested_replications.reference_containment <= nested_replications.containment_upper).all()
assert wdbc_summary['n'].eq(200).all()
assert wdbc_detail.upper.dropna().between(0, 1).all()
assert set(wdbc_summary.Model) == {'Logistic regression', 'Extra Trees', 'HistGradientBoosting'}
run.finish()
if PROFILE == 'full':
    completed = subprocess.run([sys.executable, str(ROOT / 'scripts/export_manuscript.py')],
        cwd=ROOT, text=True, capture_output=True, check=True)
    print(completed.stdout.strip())
else:
    print('Smoke output is separate; manuscript tables were not changed.')
display(pd.DataFrame(run.manifest['sections']).T)
print('External benchmark status:', run.manifest['external_benchmark'])
print('Current-run CSV files:', len(run.manifest['output_sha256']))
print('All public/synthetic sections completed. This is not a declaration of submission readiness.')

## 13. Source and revision map
- Main theory and exact/asymptotic bounds: `src/sciguard/core.py`, `groups.py`, `baselines.py`.
- Corrected DGP and MAR/MNAR mechanisms: `simulation.py`.
- All study definitions and seeds: `experiments.py`.
- Clinical cohort, interval alignment, and aggregate exports: `mimic.py`.
- Figures: `plotting.py`; manuscript tables: `scripts/export_manuscript.py`.
- Full theoretical argument and interpretation: `manuscript/main_jmlr.tex` / PDF.
- Point-by-point reviewer response and remaining evidence gaps: `docs/REVISION_RESPONSE.md`.

Primary methodological references include CCK (2013, Gaussian multiplier approximation),
Hoeffding (1963), Clopper-Pearson (1934), Holm (1979), Learn then Test (Angelopoulos et al.,
2025), and ATC (Garg et al., 2022). Bibliographic details and verified links appear in the
manuscript and `docs/SOURCE_NOTES.md`.